# Ejercicio 6: Dense Retrieval e Introducción a FAISS

## Objetivo de la práctica

Generar embeddings con sentence-transformers (SBERT, E5), e indexar documentos con FAISS 

## Parte 0: Carga del Corpus
### Actividad

1. Carga el corpus 20 Newsgroups desde sklearn.datasets.fetch_20newsgroups.
2. Limita el corpus a los primeros 2000 documentos para facilitar el procesamiento.

In [1]:
from sklearn.datasets import fetch_20newsgroups

# 1. Cargar el corpus completo
newsgroups = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes'))

# 2. Limitar a los primeros 2000 documentos
docs = newsgroups.data[:2000]
labels = newsgroups.target[:2000]

print(f"Número de documentos cargados: {len(docs)}")
print(f"Ejemplo de documento:\n{docs[0][:500]} ...")


Número de documentos cargados: 2000
Ejemplo de documento:


I am sure some bashers of Pens fans are pretty confused about the lack
of any kind of posts about the recent Pens massacre of the Devils. Actually,
I am  bit puzzled too and a bit relieved. However, I am going to put an end
to non-PIttsburghers' relief with a bit of praise for the Pens. Man, they
are killing those Devils worse than I thought. Jagr just showed you why
he is much better than his regular season stats. He is also a lot
fo fun to watch in the playoffs. Bowman should let JAgr have a ...


## Parte 2: Generación de Embeddings
### Actividad

1. Usa dos modelos de sentence-transformers. Puedes usar: `'all-MiniLM-L6-v2'` (SBERT), o `'intfloat/e5-base'` (E5). Cuando uses E5, antepon `"passage: "` a cada documento antes de codificar.
2. Genera los vectores de embeddings para todos los documentos usando el modelo seleccionado.
3. Guarda los embeddings en un array de NumPy para su posterior indexación.

In [2]:
!pip install scikit-learn
!pip install faiss-cpu
!pip install numpy
!pip install tqdm

  Obtaining dependency information for faiss-cpu from https://files.pythonhosted.org/packages/5e/9c/3018e755701023789af060ad926bf99f147fa76488ddf1e181735afe93eb/faiss_cpu-1.13.0-cp311-cp311-win_amd64.whl.metadata
   ---------------------------------------- 0.0/18.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.7 MB ? eta -:--:--
   ---------------------------------------- 0.1/18.7 MB 1.8 MB/s eta 0:00:11
   - -------------------------------------- 0.6/18.7 MB 4.9 MB/s eta 0:00:04
   -- ------------------------------------- 1.0/18.7 MB 6.5 MB/s eta 0:00:03
   --- ------------------------------------ 1.6/18.7 MB 8.0 MB/s eta 0:00:03
   ---- ----------------------------------- 2.2/18.7 MB 8.9 MB/s eta 0:00:02
   ------ --------------------------------- 3.0/18.7 MB 10.0 MB/s eta 0:00:02
   -------- ------------------------------- 3.9/18.7 MB 11.3 MB/s eta 0:00:02
   ---------- ----------------------------- 5.0/18.7 MB 12.6 MB/s eta 0:00:02
   ------------- -----------

In [3]:
!pip install sentence_transformers

  Obtaining dependency information for sentence_transformers from https://files.pythonhosted.org/packages/bb/a6/a607a737dc1a00b7afe267b9bfde101b8cee2529e197e57471d23137d4e5/sentence_transformers-5.1.2-py3-none-any.whl.metadata
  Obtaining dependency information for transformers<5.0.0,>=4.41.0 from https://files.pythonhosted.org/packages/6a/6b/2f416568b3c4c91c96e5a365d164f8a4a4a88030aa8ab4644181fdadce97/transformers-4.57.3-py3-none-any.whl.metadata
     ---------------------------------------- 0.0/44.0 kB ? eta -:--:--
     ----------------- -------------------- 20.5/44.0 kB 682.7 kB/s eta 0:00:01
     ---------------------------------------- 44.0/44.0 kB 1.1 MB/s eta 0:00:00
  Obtaining dependency information for huggingface-hub>=0.20.0 from https://files.pythonhosted.org/packages/35/f4/124858007ddf3c61e9b144107304c9152fa80b5b6c168da07d86fe583cc1/huggingface_hub-1.1.5-py3-none-any.whl.metadata
  Obtaining dependency information for fsspec>=2023.5.0 from https://files.pythonhosted.org/p

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
s3fs 2023.4.0 requires fsspec==2023.4.0, but you have fsspec 2025.10.0 which is incompatible.


### Opción A: Usar SBERT — 'all-MiniLM-L6-v2'

In [4]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Modelo SBERT
model_sbert = SentenceTransformer('all-MiniLM-L6-v2')

# Generar embeddings
embeddings_sbert = model_sbert.encode(
    docs,
    batch_size=32,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Embeddings SBERT:", embeddings_sbert.shape)

# Guardar en NumPy
np.save("embeddings_sbert.npy", embeddings_sbert)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Embeddings SBERT: (2000, 384)


### Opción B: Usar: entence-transformers/paraphrase-MiniLM-L3-v2

In [6]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("sentence-transformers/paraphrase-MiniLM-L3-v2")

embeddings = model.encode(
    docs,
    convert_to_numpy=True,
    show_progress_bar=True
)

np.save("embeddings_miniL3.npy", embeddings)


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

## Parte 3: Consulta
### Actividad

1. Escribe una consulta en lenguaje natural. Ejemplos:

    * "God, religion, and spirituality"
    * "space exploration"
    * "car maintenance"

2. Codifica la consulta utilizando el mismo modelo de embeddings. Cuando uses E5, antepon `"query: "` a la consulta.
3. Recupera los 5 documentos más relevantes con similitud coseno.
4. Muestra los textos de los documentos recuperados (puedes mostrar solo los primeros 500 caracteres de cada uno).

### Escribir una consulta

In [ ]:
query_text = "God, religion, and spirituality"


In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# -----------------------------------------------
# Cargar embeddings y modelo
# -----------------------------------------------
embeddings = np.load("embeddings_sbert.npy")          # MiniLM-L6-v2
# embeddings = np.load("embeddings_miniL3.npy")       # MiniLM-L3, si quieres usarlo

model = SentenceTransformer("all-MiniLM-L6-v2")
# model = SentenceTransformer("sentence-transformers/paraphrase-MiniLM-L3-v2")

# -----------------------------------------------
# Consulta fija
# -----------------------------------------------
query_text = "God, religion, and spirituality"

query_embedding = model.encode([query_text], convert_to_numpy=True)

# -----------------------------------------------
# Similitud coseno
# -----------------------------------------------
sims = cosine_similarity(query_embedding, embeddings)[0]

top_k = 5
top_indices = sims.argsort()[-top_k:][::-1]
top_scores = sims[top_indices]

# -----------------------------------------------
# BLOQUE A: Índices y puntajes
# -----------------------------------------------
print("===== BLOQUE A: Top 5 documentos (índice y puntaje) =====")
for rank, (idx, score) in enumerate(zip(top_indices, top_scores), start=1):
    print(f"{rank}. Índice: {idx}  |  Score: {score:.4f}")

# -----------------------------------------------
# BLOQUE B: Documentos recuperados (primeros 500 chars)
# -----------------------------------------------
print("\n===== BLOQUE B: Documentos Recuperados =====\n")

for rank, idx in enumerate(top_indices, start=1):
    print(f"--- Documento {rank} (índice {idx}) ---")
    print(docs[idx][:500].replace("\n", " "))
    print("\n")


===== BLOQUE A: Top 5 documentos (índice y puntaje) =====
1. Índice: 996  |  Score: 0.4150
2. Índice: 282  |  Score: 0.3307
3. Índice: 677  |  Score: 0.3013
4. Índice: 943  |  Score: 0.2878
5. Índice: 791  |  Score: 0.2856

===== BLOQUE B: Documentos Recuperados =====

--- Documento 1 (índice 996) ---
    Humanist, or sub-humanist? :-)


--- Documento 2 (índice 282) ---
 I didn't know God was a secular humanist...  Kent


--- Documento 3 (índice 677) ---
  (Deletion)   For me, it is a "I believe no gods exist" and a "I don't believe gods exist".   In other words, I think that statements like gods are or somehow interfere with this world are false or meaningless. In Ontology, one can fairly conclude that when "A exist" is meaningless A does not exist. Under the Pragmatic definition of truth, "A exists" is meaningless makes A exist even logically false.   A problem with such statements is that one can't disprove a subjective god by definition, and


--- Documento 4 (índice 943) ---
  Ato